# Semana 15 · ¿Cuánto tarda un planeta en dar la vuelta?

Este cuaderno acompaña a la sesión de proyecto de la semana 15: comprobar la tercera ley de Kepler con los datos reales de los planetas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Los datos

Semieje mayor (en millones de km) y periodo orbital sidéreo (en días) de los ocho planetas, según la hoja de datos planetarios de la NASA (NSSDC). Los he tomado de una reproducción de esa tabla, porque no he podido abrir la original: compruébalos en el enlace de la página de la semana.

Usamos un **diccionario** de Python: cada nombre de planeta lleva asociada una pareja de números.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

datos = {                       # semieje mayor (10^6 km), periodo sidéreo (días)
    "Mercurio": (57.91,   87.969),
    "Venus":    (108.21,  224.701),
    "Tierra":   (149.60,  365.256),
    "Marte":    (227.92,  686.980),
    "Júpiter":  (778.57,  4332.589),
    "Saturno":  (1433.53, 10759.22),
    "Urano":    (2872.46, 30685.4),
    "Neptuno":  (4495.06, 60189.0),
}

nombres = list(datos)
a_km = np.array([datos[n][0] for n in nombres])
T_d  = np.array([datos[n][1] for n in nombres])

# Pasamos a unidades astronómicas y años, tomando la Tierra como referencia
a = a_km / datos["Tierra"][0]
T = T_d / datos["Tierra"][1]

for n, ai, Ti in zip(nombres, a, T):
    print(f"{n:9s}  a = {ai:7.3f} ua   T = {Ti:8.3f} años   T²/a³ = {Ti**2 / ai**3:.4f}")

La última columna sale prácticamente 1 para todos: $T^2 = a^3$. Las pequeñas diferencias tienen varias causas: los valores de la tabla son promedios, los planetas se perturban unos a otros (Júpiter y Saturno, los más masivos, son los que más se tiran entre sí, y Saturno es el que más se aparta) y la masa de cada planeta, aunque pequeña frente a la del Sol, cambia un poco la constante.

## 2. Escala normal y escala logarítmica

En escala normal, los planetas interiores quedan amontonados en una esquina. En escala logarítmica (`plt.loglog`), cada marca del eje vale diez veces la anterior y los ocho planetas se reparten bien. Además, quedan en línea recta.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.plot(a, T, "o")
for n, ai, Ti in zip(nombres, a, T):
    ax1.annotate(n, (ai, Ti), textcoords="offset points", xytext=(4, -10), fontsize=8)
ax1.set_xlabel("semieje mayor (ua)"); ax1.set_ylabel("periodo (años)")
ax1.set_title("Escala normal"); ax1.grid(alpha=0.3)

ax2.loglog(a, T, "o")
for n, ai, Ti in zip(nombres, a, T):
    ax2.annotate(n, (ai, Ti), textcoords="offset points", xytext=(4, -10), fontsize=8)
ax2.set_xlabel("semieje mayor (ua)"); ax2.set_ylabel("periodo (años)")
ax2.set_title("Escala logarítmica en los dos ejes"); ax2.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

## 3. Medir la pendiente

Si $T = a^{k}$, entonces $\log T = k \log a$: una recta que pasa por el origen con pendiente $k$. Kepler dice que $k = 3/2$. Vamos a medirla sin suponer nada.

`np.polyfit(x, y, 1)` busca la recta $y = m\,x + n$ que mejor se ajusta a los puntos (por **mínimos cuadrados**: la que hace más pequeña la suma de los cuadrados de las distancias verticales de los puntos a la recta). Devuelve la pendiente $m$ y la ordenada en el origen $n$.

In [ ]:
log_a = np.log10(a)
log_T = np.log10(T)
m, n = np.polyfit(log_a, log_T, 1)
print(f"Pendiente: {m:.4f}   (Kepler: 1.5)")
print(f"Ordenada en el origen: {n:.4f}   (debería ser 0, porque la Tierra tiene a = 1 y T = 1)")

x = np.linspace(log_a.min() - 0.2, log_a.max() + 0.2, 10)
plt.plot(log_a, log_T, "o", label="planetas")
plt.plot(x, m * x + n, label=f"recta ajustada, pendiente {m:.3f}")
plt.xlabel("log₁₀ a"); plt.ylabel("log₁₀ T"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Ejercicio.** Usa la tercera ley para calcular el periodo de cada planeta a partir de su semieje mayor, $T = a^{3/2}$, y el error en porcentaje respecto al periodo real.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
T_kepler = a ** 1.5
for nombre, Tk, Tr in zip(nombres, T_kepler, T):
    print(f"{nombre:9s}  Kepler: {Tk:8.3f} años   real: {Tr:8.3f} años   error: {100 * (Tk - Tr) / Tr:+.2f} %")

## 4. Más allá de los planetas

La ley vale para cualquier cosa que gire alrededor del Sol. Algunos semiejes mayores aproximados (de memoria; compruébalos si quieres usarlos para algo más que practicar):

- Ceres, el mayor cuerpo del cinturón de asteroides: 2,77 ua.
- Plutón: 39,5 ua.
- Cometa Halley: 17,8 ua.
- Sedna, un objeto muy lejano descubierto en 2003: unas 500 ua (su órbita no se conoce con mucha precisión).

**Ejercicio.** Calcula sus periodos y añádelos a la gráfica logarítmica con otro color.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
otros = {"Ceres": 2.77, "Plutón": 39.5, "Halley": 17.8, "Sedna": 500}
plt.loglog(a, T, "o", label="planetas")
for nombre, ao in otros.items():
    To = ao ** 1.5
    print(f"{nombre:7s}  a = {ao:6.2f} ua   T ≈ {To:9.1f} años")
    plt.loglog(ao, To, "s", color="tab:red")
    plt.annotate(nombre, (ao, To), textcoords="offset points", xytext=(4, -10), fontsize=8)
plt.xlabel("semieje mayor (ua)"); plt.ylabel("periodo (años)"); plt.grid(alpha=0.3, which="both")
plt.legend(); plt.show()

## 5. Experimenta

1. La velocidad media de un planeta es la longitud de su órbita dividida por el periodo: $v \approx 2\pi a / T$. Calcúlala en km/s para los ocho planetas y dibuja $v$ frente a $a$ en escala logarítmica. ¿Qué pendiente esperas? ¿Cuál sale?
2. Con las fechas de las oposiciones de Marte de la sesión 4, calcula el periodo sidéreo en Python. Usa `datetime.date(2022, 12, 8)` y resta fechas para contar los días.

In [ ]:
# Tu experimento aquí